# 集成学习第五课：GBDT

## 本课目标

GBDT 的核心不是背公式，而是看懂这条线：

先有一个旧模型：

$$
F_{t-1}(x)
$$

旧模型还有没预测好的部分：

$$
y_i - F_{t-1}(x_i)
$$

新树就去学习这个没预测好的部分。

所以回归版 GBDT 可以先记成：

一句话理解：每一轮训练一棵树去拟合当前残差。

本课只讲回归版 GBDT。分类版后面再学。

## 1. GBDT 是什么？

GBDT 是 Gradient Boosting Decision Tree。

它由很多棵树加起来：

$$
F_T(x)
=
F_0(x)
+
\nu f_1(x)
+
\nu f_2(x)
+
\cdots
+
\nu f_T(x)
$$

其中：

- $F_0(x)$：第 $0$ 轮模型，也就是还没有训练任何树之前的初始预测。
- $f_t(x)$：第 $t$ 轮新训练出来的那一棵小树，用来修正前面模型的错误。
- $\nu$：学习率，用来控制每一棵小树对最终模型的影响力度。
- $T$：总共训练多少棵小树。

注意区分：

- $F_t(x)$：第 $t$ 轮结束后的整体模型。
- $f_t(x)$：第 $t$ 轮新加入的单棵树。

所以：

$$
F_t(x)
\ne
f_t(x)
$$

$F_t(x)$ 是很多棵树加起来之后的总模型，$f_t(x)$ 只是这一轮新加进去的一棵树。

也可以写成递推形式：

$$
F_t(x)
=
F_{t-1}(x)
+
\nu f_t(x)
$$

说人话：新模型等于旧模型加上一点新树的修正。

## 2. 先看损失函数

回归问题常用平方误差。

为了求导方便，通常写成：

$$
L(y,F)
=
\frac{1}{2}
\left(y-F\right)^2
$$

这里：

- $y$：真实值。
- $F$：模型预测值。

前面的：

$$
\frac{1}{2}
$$

不是重点，只是为了让求导时把平方前面的 $2$ 抵消掉。

如果有 $m$ 个样本，总损失就是：

$$
J(F)
=
\sum_{i=1}^{m}
\frac{1}{2}
\left(y_i-F(x_i)\right)^2
$$

## 3. 初始模型为什么是均值？

一开始还没有树，GBDT 先用一个常数预测所有样本。

假设这个常数是：

$$
c
$$

那么每个样本都预测为：

$$
F_0(x_i)=c
$$

此时总损失是：

$$
J(c)
=
\sum_{i=1}^{m}
\frac{1}{2}
\left(y_i-c\right)^2
$$

为了找到最好的 $c$，对 $c$ 求导。

单个样本的损失是：

$$
\frac{1}{2}
\left(y_i-c\right)^2
$$

对 $c$ 求导：

$$
\frac{d}{dc}
\left[
\frac{1}{2}
\left(y_i-c\right)^2
\right]
=
c-y_i
$$

所以总损失的导数是：

$$
\frac{dJ(c)}{dc}
=
\sum_{i=1}^{m}
\left(c-y_i\right)
$$

令导数等于 $0$：

$$
\sum_{i=1}^{m}
\left(c-y_i\right)
=
0
$$

展开：

$$
mc-\sum_{i=1}^{m}y_i
=
0
$$

所以：

$$
c
=
\frac{1}{m}
\sum_{i=1}^{m}y_i
$$

也就是：

$$
F_0(x)=\bar{y}
$$

说人话：

说人话：如果一开始只能猜一个常数，平方误差下最好的猜法就是猜平均值。

## 4. 为什么下一棵树要拟合残差？

第 $t$ 轮开始前，旧模型已经给出预测：

$$
F_{t-1}(x_i)
$$

现在想加一棵新树：

$$
f_t(x)
$$

如果暂时不考虑学习率，更新后预测是：

$$
F_t(x_i)
=
F_{t-1}(x_i)
+
f_t(x_i)
$$

我们希望新预测尽量接近真实值：

$$
F_t(x_i)\approx y_i
$$

代入：

$$
F_{t-1}(x_i)
+
f_t(x_i)
\approx
y_i
$$

移项：

$$
f_t(x_i)
\approx
y_i-F_{t-1}(x_i)
$$

右边就是残差：

$$
r_i^{(t)}
=
y_i-F_{t-1}(x_i)
$$

所以：

$$
f_t(x_i)
\approx
r_i^{(t)}
$$

这就是为什么新树要拟合残差。

分情况理解：

$$
r_i^{(t)}>0
$$

表示旧模型预测低了，新树应该输出正数，把预测往上补。

$$
r_i^{(t)}<0
$$

表示旧模型预测高了，新树应该输出负数，把预测往下拉。

$$
r_i^{(t)}=0
$$

表示旧模型刚好预测对了，新树不需要修正这个样本。

## 5. 残差为什么也是负梯度？

GBDT 里的 Gradient 指的是梯度。

平方误差是：

$$
L(y,F)
=
\frac{1}{2}
\left(y-F\right)^2
$$

把 $F$ 当作变量，对 $F$ 求导。

先令：

$$
u=y-F
$$

那么：

$$
L
=
\frac{1}{2}u^2
$$

对 $u$ 求导：

$$
\frac{dL}{du}
=
u
$$

而：

$$
u=y-F
$$

所以：

$$
\frac{du}{dF}
=
-1
$$

根据链式法则：

$$
\frac{dL}{dF}
=
\frac{dL}{du}
\cdot
\frac{du}{dF}
=
u\cdot(-1)
=
-(y-F)
$$

所以：

$$
\frac{dL}{dF}
=
F-y
$$

负梯度是：

$$
-
\frac{dL}{dF}
=
y-F
$$

而残差也是：

$$
r
=
y-F
$$

因此在平方误差下，负梯度等于残差。

所以回归版 GBDT 说“拟合残差”，本质上是在沿着让损失下降的方向修正模型。

## 6. 一棵树的叶子值为什么取残差均值？

GBDT 每一轮要训练一棵回归树去拟合残差。

假设某个叶子节点里有一批样本，记作：

$$
R
$$

这片叶子对所有样本输出同一个值：

$$
c
$$

为了让这片叶子尽量拟合残差，要最小化：

$$
J(c)
=
\sum_{i\in R}
\frac{1}{2}
\left(r_i-c\right)^2
$$

对 $c$ 求导：

$$
\frac{dJ(c)}{dc}
=
\sum_{i\in R}
\left(c-r_i\right)
$$

令导数等于 $0$：

$$
\sum_{i\in R}
\left(c-r_i\right)
=
0
$$

展开：

$$
|R|c
-
\sum_{i\in R}r_i
=
0
$$

所以：

$$
c
=
\frac{1}{|R|}
\sum_{i\in R}
r_i
$$

也就是说：叶子输出值等于这个叶子里残差的平均值。

## 7. 完整手算例子：三轮 GBDT

我们用一个一维回归数据集：

| 样本 | $x_i$ | $y_i$ |
|---:|---:|---:|
| $1$ | $1$ | $2$ |
| $2$ | $2$ | $4$ |
| $3$ | $3$ | $6$ |
| $4$ | $4$ | $8$ |
| $5$ | $5$ | $10$ |
| $6$ | $6$ | $12$ |

为了手算简单，弱学习器只用树桩。

树桩就是只切一刀：

$$
f(x)
=
\begin{cases}
c_1, & x\le s \\[6pt]
c_2, & x>s
\end{cases}
$$

其中：

- $s$：切分点。
- $c_1$：左叶子输出。
- $c_2$：右叶子输出。

候选切分点是：

$$
1.5,\ 2.5,\ 3.5,\ 4.5,\ 5.5
$$

学习率设为：

$$
\nu=0.5
$$

### 第 $0$ 步：初始预测

标签平均值：

$$
\bar{y}
=
\frac{2+4+6+8+10+12}{6}
=
7
$$

所以：

$$
F_0(x)=7
$$

初始预测：

$$
[7,\ 7,\ 7,\ 7,\ 7,\ 7]
$$

初始残差：

$$
r_i^{(1)}
=
y_i-F_0(x_i)
$$

逐个算：

$$
2-7=-5
$$

$$
4-7=-3
$$

$$
6-7=-1
$$

$$
8-7=1
$$

$$
10-7=3
$$

$$
12-7=5
$$

所以：

$$
r^{(1)}
=
[-5,\ -3,\ -1,\ 1,\ 3,\ 5]
$$

### 第 $1$ 轮：训练第一棵树

第 $1$ 轮要拟合残差：

$$
[-5,\ -3,\ -1,\ 1,\ 3,\ 5]
$$

候选切分点是：

$$
1.5,\ 2.5,\ 3.5,\ 4.5,\ 5.5
$$

每一个切分点都对应一棵树桩：

$$
f_1(x)
=
\begin{cases}
c_1, & x\le s \\[6pt]
c_2, & x>s
\end{cases}
$$

其中：

- \(c_1\)：左叶子里残差的平均值。
- \(c_2\)：右叶子里残差的平均值。

然后计算这棵树对残差的平方误差。

### 为什么要比较残差平方误差？

第 \(1\) 轮的目标是：训练一棵树去拟合当前残差。

也就是希望树的输出：

$$
f_1(x_i)
$$

尽量接近当前残差：

$$
r_i^{(1)}
$$

对第 \(i\) 个样本来说，树输出和残差之间的差是：

$$
r_i^{(1)} - f_1(x_i)
$$

差越小，说明这棵树对这个样本的残差拟合得越好。

为了衡量一棵树对所有样本的残差拟合得怎么样，就把所有差距平方后加起来：

$$
\sum_{i=1}^{m}
\left(
r_i^{(1)} - f_1(x_i)
\right)^2
$$

这就是残差平方误差。

所以比较不同切分点，本质上是在问：

哪一个切分点得到的树，最能拟合当前残差？

因此我们选择残差平方误差最小的切分点。

### 候选切分点比较

| 切分点 \(s\) | 左叶子残差 | 左叶子输出 \(c_1\) | 右叶子残差 | 右叶子输出 \(c_2\) | 残差平方误差 |
|---:|---|---:|---|---:|---:|
| \(1.5\) | \([-5]\) | \(-5\) | \([-3,-1,1,3,5]\) | \(1\) | \(40\) |
| \(2.5\) | \([-5,-3]\) | \(-4\) | \([-1,1,3,5]\) | \(2\) | \(22\) |
| \(3.5\) | \([-5,-3,-1]\) | \(-3\) | \([1,3,5]\) | \(3\) | \(16\) |
| \(4.5\) | \([-5,-3,-1,1]\) | \(-2\) | \([3,5]\) | \(4\) | \(22\) |
| \(5.5\) | \([-5,-3,-1,1,3]\) | \(-1\) | \([5]\) | \(5\) | \(40\) |

残差平方误差最小的是：

$$
16
$$

对应切分点：

$$
s=3.5
$$

所以第 $1$ 轮选择：

$$
f_1(x)
=
\begin{cases}
-3, & x\le 3.5 \\[6pt]
3, & x>3.5
\end{cases}
$$

为什么 \(s=3.5\) 的平方误差是 \(16\)？

左边残差是：

$$
-5,\ -3,\ -1
$$

左叶子输出是平均值：

$$
c_1
=
\frac{-5-3-1}{3}
=
-3
$$

左边平方误差：

$$
(-5+3)^2+(-3+3)^2+(-1+3)^2
=
4+0+4
=
8
$$

右边残差是：

$$
1,\ 3,\ 5
$$

右叶子输出是平均值：

$$
c_2
=
\frac{1+3+5}{3}
=
3
$$

右边平方误差：

$$
(1-3)^2+(3-3)^2+(5-3)^2
=
4+0+4
=
8
$$

所以总平方误差：

$$
8+8=16
$$

这棵树对六个样本的输出是：

$$
[-3,\ -3,\ -3,\ 3,\ 3,\ 3]
$$

更新：

$$
F_1(x)
=
F_0(x)
+
\nu f_1(x)
$$

因为：

$$
\nu=0.5
$$

所以左边样本：

$$
7+0.5(-3)=5.5
$$

右边样本：

$$
7+0.5(3)=8.5
$$

新预测：

$$
[5.5,\ 5.5,\ 5.5,\ 8.5,\ 8.5,\ 8.5]
$$

新残差(真实值-本轮预测)：

$$
[-3.5,\ -1.5,\ 0.5,\ -0.5,\ 1.5,\ 3.5]
$$


### 第 $2$ 轮：训练第二棵树

第 $2$ 轮拟合新残差：

$$
[-3.5,\ -1.5,\ 0.5,\ -0.5,\ 1.5,\ 3.5]
$$

枚举切分点后，最优切分点可以取：

$$
s=2.5
$$

左边残差：

$$
-3.5,\ -1.5
$$

左叶子输出：

$$
c_1
=
\frac{-3.5-1.5}{2}
=
-2.5
$$

右边残差：

$$
0.5,\ -0.5,\ 1.5,\ 3.5
$$

右叶子输出：

$$
c_2
=
\frac{0.5-0.5+1.5+3.5}{4}
=
1.25
$$

所以：

$$
f_2(x)
=
\begin{cases}
-2.5, & x\le 2.5 \\[6pt]
1.25, & x>2.5
\end{cases}
$$

更新：

$$
F_2(x)
=
F_1(x)
+
0.5f_2(x)
$$

逐个样本的新预测是：

$$
[4.25,\ 4.25,\ 6.125,\ 9.125,\ 9.125,\ 9.125]
$$

新残差：

$$
[-2.25,\ -0.25,\ -0.125,\ -1.125,\ 0.875,\ 2.875]
$$

### 第 $3$ 轮：训练第三棵树

第 $3$ 轮拟合新残差：

$$
[-2.25,\ -0.25,\ -0.125,\ -1.125,\ 0.875,\ 2.875]
$$

枚举切分点后，最优切分点是：

$$
s=4.5
$$

左边残差：

$$
-2.25,\ -0.25,\ -0.125,\ -1.125
$$

左叶子输出：

$$
c_1
=
\frac{-2.25-0.25-0.125-1.125}{4}
=
-0.9375
$$

右边残差：

$$
0.875,\ 2.875
$$

右叶子输出：

$$
c_2
=
\frac{0.875+2.875}{2}
=
1.875
$$

所以：

$$
f_3(x)
=
\begin{cases}
-0.9375, & x\le 4.5 \\[6pt]
1.875, & x>4.5
\end{cases}
$$

更新：

$$
F_3(x)
=
F_2(x)
+
0.5f_3(x)
$$

新预测：

$$
[3.78125,\ 3.78125,\ 5.65625,\ 8.65625,\ 10.0625,\ 10.0625]
$$

真实值是：

$$
[2,\ 4,\ 6,\ 8,\ 10,\ 12]
$$

可以看到，预测正在一轮一轮靠近真实值。

In [ ]:
import numpy as np
import pandas as pd

x = np.array([1, 2, 3, 4, 5, 6], dtype=float)
y = np.array([2, 4, 6, 8, 10, 12], dtype=float)
learning_rate = 0.5

def best_stump_for_residual(x, residual):
    thresholds = [(x[i] + x[i + 1]) / 2 for i in range(len(x) - 1)]
    rows = []
    for threshold in thresholds:
        left = x <= threshold
        right = ~left
        left_value = residual[left].mean()
        right_value = residual[right].mean()
        pred = np.where(left, left_value, right_value)
        sse = np.sum((residual - pred) ** 2)
        rows.append({
            "切分点": threshold,
            "左叶子输出": left_value,
            "右叶子输出": right_value,
            "残差平方误差": sse,
            "树输出": pred,
        })
    return pd.DataFrame(rows).sort_values("残差平方误差").reset_index(drop=True)

prediction = np.ones_like(y) * y.mean()
round_summaries = []
detail_tables = []

for round_index in range(1, 4):
    residual = y - prediction
    candidates = best_stump_for_residual(x, residual)
    best = candidates.iloc[0]
    tree_output = best["树输出"]
    new_prediction = prediction + learning_rate * tree_output

    round_summaries.append({
        "轮数": round_index,
        "切分点": best["切分点"],
        "左叶子输出": best["左叶子输出"],
        "右叶子输出": best["右叶子输出"],
        "残差平方误差": best["残差平方误差"],
    })

    detail_tables.append(pd.DataFrame({
        "轮数": round_index,
        "x": x,
        "y": y,
        "本轮开始预测": prediction,
        "本轮残差": residual,
        "树输出": tree_output,
        "学习率乘树输出": learning_rate * tree_output,
        "本轮更新后预测": new_prediction,
        "更新后残差": y - new_prediction,
    }))

    prediction = new_prediction

summary_table = pd.DataFrame(round_summaries)
detail_table = pd.concat(detail_tables, ignore_index=True)

summary_table, detail_table

## 8. 学习率

GBDT 的更新公式是：

$$
F_t(x)
=
F_{t-1}(x)
+
\nu f_t(x)
$$

如果：

$$
\nu=1
$$

每棵树的输出完整加入模型。

如果：

$$
\nu=0.1
$$

每棵树只贡献自己输出的十分之一。

说人话：学习率越小，每一棵树修正得越谨慎。

所以通常：学习率越小，需要更多树。

## 9. sklearn 实战：GBDT 回归

sklearn 中常用：

`GradientBoostingRegressor`

下面用一个小数据集看 GBDT 如何拟合非线性回归关系。

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(22)
X = np.linspace(0, 6, 80).reshape(-1, 1)
y_reg = np.sin(X[:, 0]) + 0.2 * X[:, 0] + rng.normal(0, 0.08, size=80)

model = GradientBoostingRegressor(
    n_estimators=80,
    learning_rate=0.08,
    max_depth=2,
    random_state=22,
)
model.fit(X, y_reg)
pred = model.predict(X)

mean_squared_error(y_reg, pred)

常见参数：

| 参数 | 含义 |
|---|---|
| `n_estimators` | 树的数量 |
| `learning_rate` | 学习率 |
| `max_depth` | 每棵树的最大深度 |
| `subsample` | 每轮使用多少比例样本 |
| `random_state` | 随机种子 |

GBDT 调参时最常先看：

$$
n\_estimators
$$

$$
learning\_rate
$$

$$
max\_depth
$$

## 10. 本课小结

GBDT 的核心流程：

1. 先给初始预测。即求平均数。

2. 计算当前残差。找到候选切分点，根据这个切分点划分样本，计算每个叶子输出残差均值，残差减去叶子输出的平方计算残差平方误差和。

3. 训练一棵树拟合残差。

4. 把这棵树乘学习率后加到旧模型上。注意前面要乘学习率。用原始数据减去新模型得到新的残差。

5. 利用新的残差去重复多轮上述步骤，继续寻找最佳切分点做切分然后得到模型。

三条关键数学来源：

第一，初始值为什么是均值：平方误差下，最优常数预测是 $\bar{y}$。

第二，为什么拟合残差：

$$
F_{t-1}(x_i)+f_t(x_i)\approx y_i
\Rightarrow
f_t(x_i)\approx y_i-F_{t-1}(x_i)
$$

第三，为什么叶子输出取残差均值：一个叶子里，最小平方误差的常数就是残差均值。

一句话总结：GBDT 的每一棵新树，都在修正前面模型还没预测好的部分。